# Evaluate_3D

3D waterfalls of the reconstructed **V/C''** curves (`z = V/C''` vs `x = log(K/S0)`), stacked along a third axis. All curves are rebuilt **exactly** from the saved `fit_results_*.csv` artefacts (same `recon_theta` + `vc2_curve` machinery verified in `Evaluate_samples`); `AugmentedLagrangian.ipynb` is not modified.

Notation: **`T` = maturity** (the fixed expiry date); **`t` = time to maturity** (years remaining, `t = maturity − start_date`).

**Plot 1 — fix maturity `T` (= expiry 20110331), vary the rest.** Within one maturity the 10 start dates each have a different time to maturity `t`, so the 3rd axis is `t`. One figure per pipeline (the four 20110331 tests).

**Plot 2 — fix time to maturity `t`, vary maturity `T`.** No single maturity folder holds a constant `t`, so we look **across maturities** in a pipeline root: for a target `t*` we take, from each maturity, the start date whose `t` is closest to `t*` (nearest-per-maturity). The 3rd axis is then **maturity `T`** — showing how the smile at ~constant time to maturity changes across maturities. Produced for `t* = 0.25, 0.5, 1` (three figures).

`z` is drawn as `log10(V/C'')` throughout, since V/C'' spans several orders of magnitude (matching the semilog view of the 2D compare plots).

In [5]:
# ── setup ───────────────────────────────────────────────────
# Load the EXACT definitions from AugmentedLagrangian.ipynb (calculate_J, generate_C2K,
# generate_call_prices, set_J_objective/normalize) by executing its definition cells (0..14).
import json, glob, os, re
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.colors import Normalize
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 (enables 3d projection)

_ALNB = json.load(open("AugmentedLagrangian.ipynb"))
for _i in range(0, 15):
    exec(compile("".join(_ALNB["cells"][_i]["source"]), f"<AL cell {_i}>", "exec"))
set_J_objective(2)
set_J_normalize(True)

N_GRID = 400
MATURITY = 20110331          # T : the fixed maturity (expiry) for plot 1
OUT_DIR = "evaluation_3d"
os.makedirs(OUT_DIR, exist_ok=True)

# Plot 1: the four fixed-maturity-20110331 tests (same folders as Evaluate_samples)
PIPELINES = [
    ("sig_nu + sig_only",        "testing_2011_2012_signu_vs_sigonly/20110331_20260627_000143"),
    ("sig_only [1]",             "testing_2011_2012_signu_vs_sigonly/20110331_20260627_024000"),
    ("sig_only [2] + sig_nu",    "testing_2011_2022_sigonly/20110331_20260628_224809"),
    ("sig_only_LKbar + sig_nu",  "testing_2011_2012_sigonlyLU/20110331_20260627_095627"),
]

# Plot 2: one pipeline root spanning MANY maturities (so many t values exist -> we can fix t).
ROOT2       = "testing_2011_2022_sigonly"   # ~many maturity subfolders (single sig_only mode)
TARGET_TS   = [0.25, 0.5, 1.0]               # times to maturity t* to hold ~constant (3 figures)
WHICH_CURVE = "final"                        # "final" (optimized) or "init" (arb-free input)
print("setup ok | plot1:", [p[0] for p in PIPELINES], "| plot2 root:", ROOT2, "t*=", TARGET_TS)

setup ok | plot1: ['sig_nu + sig_only', 'sig_only [1]', 'sig_only [2] + sig_nu', 'sig_only_LKbar + sig_nu'] | plot2 root: testing_2011_2022_sigonly t*= [0.25, 0.5, 1.0]


In [6]:
# ── reconstruction helpers (verified in Evaluate_samples) ─────────────────────
def recon_theta(df, block):
    """theta = cat(nus1, sigs1, nus2, sigs2) and (R1, R2) from a param_init/param_final block."""
    b = df[df["record_type"] == block]
    L = b[b["side"] == "L"].sort_values("idx"); R = b[b["side"] == "R"].sort_values("idx")
    nus1  = torch.tensor(L["nu"].to_numpy(dtype=float)); sigs1 = torch.tensor(L["sigma"].to_numpy(dtype=float))
    nus2  = torch.tensor(R["nu"].to_numpy(dtype=float)); sigs2 = torch.tensor(R["sigma"].to_numpy(dtype=float))
    return torch.cat([nus1, sigs1, nus2, sigs2]), len(L), len(R)

def vc2_curve(R1, R2, S0, theta, kmin, kmax, n_grid=N_GRID):
    """EXACT copy of compare_start_dates._VC2_curve: (log-moneyness, V/C'') on a fine K grid."""
    K  = np.linspace(kmin, kmax, n_grid)
    Kt = torch.tensor(K, dtype=torch.float64)
    with torch.no_grad():
        _, _, _, c1, c2 = calculate_J(R1, R2, S0, theta)
        c2k = generate_C2K(Kt, R1, R2, S0, theta, c1, c2).cpu().numpy()
        Ck  = generate_call_prices(Kt, R1, R2, S0, theta, c1, c2).cpu().numpy()
    V = Ck - np.maximum(S0 - K, 0.0)
    m = c2k > 0
    return np.log(K[m] / S0), np.clip(V[m] / c2k[m], 1e-30, None)

def _tmap(folder):
    """start_date -> t (time to maturity, years) from the folder's manifest(s)."""
    T = {}
    for man in glob.glob(os.path.join(folder, "manifest_*.csv")):
        md = pd.read_csv(man)
        T.update(dict(zip(md["start_date"].astype(int), md["T"].astype(float))))
    return T

def _curve_from_csv(f, which="final"):
    """Reconstruct one (x=log-moneyness, vc2, start_date, S0, ...) from a fit_results CSV."""
    df = pd.read_csv(f)
    m = df[df["record_type"] == "meta"]; meta = dict(zip(m["key"], m["value"]))
    S0 = float(meta["S0"]); sd = int(float(meta["dataset_start_date"]))
    mk = df[df["record_type"] == "market"]; strikes = mk["strike"].to_numpy(dtype=float)
    kmin, kmax = float(strikes.min()), float(strikes.max())
    block = "param_final" if which == "final" else "param_init"
    th, R1, R2 = recon_theta(df, block)
    x, vc2 = vc2_curve(R1, R2, S0, th, kmin, kmax)
    return dict(start_date=sd, S0=S0, x=x, vc2=vc2,
                mode=str(meta["optimize_mode"]),
                true_reduce=float(meta["true_roughness_reduction_pct"]))

def load_maturity_folder(folder, which="final"):
    """Winner curve per start date in one maturity folder, each tagged with its t. For plot 1."""
    tmap = _tmap(folder)
    per = {}
    for f in sorted(glob.glob(os.path.join(folder, "fit_results_*.csv"))):
        df = pd.read_csv(f, nrows=25); mm = df[df.record_type == "meta"].set_index("key")["value"]
        sd = int(float(mm["dataset_start_date"])); tr = float(mm["true_roughness_reduction_pct"])
        per.setdefault(sd, []).append((tr, f))
    recs = []
    for sd in sorted(per):
        _, f = max(per[sd], key=lambda t: t[0])   # winner = max true reduction
        r = _curve_from_csv(f, which); r["t"] = float(tmap.get(sd, np.nan))
        recs.append(r)
    return recs

In [7]:
# ── PLOT 1: fix maturity T=20110331, 3rd axis = t (time to maturity) ────────────
def plot3d_fixed_maturity(name, recs, outpath, which="final"):
    recs = [r for r in recs if r["x"].size]
    recs.sort(key=lambda r: r["t"])
    ts = np.array([r["t"] for r in recs])
    norm = Normalize(vmin=ts.min(), vmax=ts.max()); cmap = cm.viridis
    fig = plt.figure(figsize=(13, 9)); ax = fig.add_subplot(111, projection="3d")
    for r in recs:
        z = np.log10(r["vc2"])
        ax.plot(r["x"], np.full_like(r["x"], r["t"]), z, color=cmap(norm(r["t"])), lw=1.4)
    ax.set_xlabel("log(K / S0)", labelpad=10)
    ax.set_ylabel("t  (time to maturity, years)", labelpad=10)
    ax.set_zlabel("log10( V / C'' )", labelpad=8)
    ax.set_title(f"V/C'' surface ({which}) | fixed maturity T={MATURITY} | {name}\n"
                 f"3rd axis = t (time to maturity), {len(recs)} start dates",
                 fontsize=12, fontweight="bold")
    ax.view_init(elev=22, azim=-60)
    sm = cm.ScalarMappable(norm=norm, cmap=cmap); sm.set_array([])
    fig.colorbar(sm, ax=ax, shrink=0.6, pad=0.10, label="t (years)")
    fig.tight_layout()
    fig.savefig(outpath, dpi=140, bbox_inches="tight")
    fig.savefig(os.path.splitext(outpath)[0] + ".svg", bbox_inches="tight")
    plt.close(fig)
    print(f"[plot1] {name} -> {outpath} ({len(recs)} curves)")

def _slug(s):
    return re.sub(r"[^0-9a-zA-Z]+", "_", s).strip("_").lower()

for name, folder in PIPELINES:
    if not os.path.isdir(folder):
        print(f"[skip] {name}: {folder} not found"); continue
    recs = load_maturity_folder(folder, which=WHICH_CURVE)
    plot3d_fixed_maturity(name, recs, os.path.join(OUT_DIR, f"fixed_maturity_{_slug(name)}.png"), which=WHICH_CURVE)
print("plot 1 done")

[plot1] sig_nu + sig_only -> evaluation_3d/fixed_maturity_sig_nu_sig_only.png (10 curves)
[plot1] sig_only [1] -> evaluation_3d/fixed_maturity_sig_only_1.png (10 curves)
[plot1] sig_only [2] + sig_nu -> evaluation_3d/fixed_maturity_sig_only_2_sig_nu.png (10 curves)
[plot1] sig_only_LKbar + sig_nu -> evaluation_3d/fixed_maturity_sig_only_lkbar_sig_nu.png (10 curves)
plot 1 done


## Plot 2 — fix time to maturity `t`, vary maturity `T` (across maturities)

Because `t` slides with the start date inside any one maturity folder, a constant-`t` slice must be assembled **across maturity subfolders** of a pipeline root. Method used below (nearest-per-maturity):

1. Walk every maturity subfolder under `ROOT2`; for each `(maturity, start_date)` sample read its `t` from the manifest.
2. For each **maturity** `T`, keep the single start date whose `t` is closest to `t*` (within a guard band).
3. Reconstruct that sample's V/C'' curve and place it at 3rd-axis = maturity `T`, coloured by maturity.

Result: at ~constant time to maturity `t*`, how the V/C'' smile changes across maturities. Each curve keeps its *actual* `t` (title reports the spread around `t*`). Produced for `t* = 0.25, 0.5, 1`.

In [8]:
# ── PLOT 2: fix time to maturity t~=t*, 3rd axis = maturity T (expiry) ────────────
def collect_constant_t(root, target_t, band=0.10, which="final"):
    """Nearest-per-maturity: one curve per maturity subfolder whose t is closest to target_t
    (and within +/- band years). Returns list sorted by maturity (expiry)."""
    picks = []
    for folder in sorted(glob.glob(os.path.join(root, "*"))):
        if not os.path.isdir(folder):
            continue
        tmap = _tmap(folder)
        per = {}
        for f in glob.glob(os.path.join(folder, "fit_results_*.csv")):
            try:
                df = pd.read_csv(f, nrows=25); mm = df[df.record_type == "meta"].set_index("key")["value"]
                sd = int(float(mm["dataset_start_date"])); tr = float(mm["true_roughness_reduction_pct"])
                mat = int(float(mm["dataset_expiry_date"]))
            except Exception:
                continue
            per.setdefault(sd, []).append((tr, f, mat))
        cands = []
        for sd, lst in per.items():
            t = tmap.get(sd, np.nan)
            if not np.isfinite(t):
                continue
            _, f, mat = max(lst, key=lambda z: z[0])
            cands.append((abs(t - target_t), t, sd, f, mat))
        if not cands:
            continue
        cands.sort()
        dt, t, sd, f, mat = cands[0]
        if dt > band:            # no start date near enough to target t in this maturity
            continue
        try:
            r = _curve_from_csv(f, which); r["t"] = float(t); r["maturity"] = mat
            picks.append(r)
        except Exception as e:
            print(f"  skip {os.path.basename(folder)}: {e}")
    picks.sort(key=lambda r: r["maturity"])
    return picks

def plot3d_fixed_t(root, target_t, picks, outpath, which="final"):
    picks = [r for r in picks if r["x"].size]
    if not picks:
        print(f"[plot2] no curves near t={target_t}"); return
    y = np.arange(len(picks))                     # maturity ordinal (even spacing, dated ticks)
    labels = [str(r["maturity"]) for r in picks]
    ts = np.array([r["t"] for r in picks])
    norm = Normalize(vmin=y.min(), vmax=y.max()); cmap = cm.plasma
    fig = plt.figure(figsize=(14, 9)); ax = fig.add_subplot(111, projection="3d")
    for j, r in enumerate(picks):
        z = np.log10(r["vc2"])
        ax.plot(r["x"], np.full_like(r["x"], y[j]), z, color=cmap(norm(y[j])), lw=1.3)
    ax.set_xlabel("log(K / S0)", labelpad=10)
    ax.set_ylabel("maturity  T  (expiry date)", labelpad=18)
    ax.set_zlabel("log10( V / C'' )", labelpad=8)
    step = max(1, len(picks) // 12)
    ax.set_yticks(y[::step]); ax.set_yticklabels(labels[::step], fontsize=7, rotation=-15)
    ax.set_title(f"V/C'' surface ({which}) | fixed time to maturity t≈{target_t:.3g}y | root={root}\n"
                 f"3rd axis = maturity T; {len(picks)} maturities, actual t in "
                 f"[{ts.min():.3g}, {ts.max():.3g}]y",
                 fontsize=12, fontweight="bold")
    ax.view_init(elev=22, azim=-60)
    fig.tight_layout()
    fig.savefig(outpath, dpi=140, bbox_inches="tight")
    fig.savefig(os.path.splitext(outpath)[0] + ".svg", bbox_inches="tight")
    plt.close(fig)
    print(f"[plot2] t~{target_t} -> {outpath} ({len(picks)} curves; actual t {ts.min():.3g}..{ts.max():.3g})")

if os.path.isdir(ROOT2):
    for target_t in TARGET_TS:
        picks = collect_constant_t(ROOT2, target_t, band=0.10, which=WHICH_CURVE)
        plot3d_fixed_t(ROOT2, target_t, picks,
                       os.path.join(OUT_DIR, f"fixed_t_{_slug(ROOT2)}_t{str(target_t).replace('.','p')}.png"),
                       which=WHICH_CURVE)
else:
    print(f"[skip] plot 2: root {ROOT2} not found")
print("plot 2 done. Images in", OUT_DIR)

/var/folders/t5/f551fpw93k53k0ygfthd91040000gn/T/ipykernel_16412/3088856320.py:62: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all Axes decorations.
  fig.tight_layout()


[plot2] t~0.25 -> evaluation_3d/fixed_t_testing_2011_2022_sigonly_t0p25.png (34 curves; actual t 0.218..0.266)


/var/folders/t5/f551fpw93k53k0ygfthd91040000gn/T/ipykernel_16412/3088856320.py:62: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all Axes decorations.
  fig.tight_layout()


[plot2] t~0.5 -> evaluation_3d/fixed_t_testing_2011_2022_sigonly_t0p5.png (20 curves; actual t 0.437..0.552)


/var/folders/t5/f551fpw93k53k0ygfthd91040000gn/T/ipykernel_16412/3088856320.py:62: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all Axes decorations.
  fig.tight_layout()


[plot2] t~1.0 -> evaluation_3d/fixed_t_testing_2011_2022_sigonly_t1p0.png (18 curves; actual t 0.992..1.09)
plot 2 done. Images in evaluation_3d
